In [ ]:
import pyorbital
from pyorbital.orbital import Orbital
import datetime as dt
from matplotlib import colormaps as cmap
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import numpy as np
import pandas as pd
import bisect
import uuid
from enum import Enum

from fame import *
import copy

In [ ]:
# Behind the scenes, this pulls from Celestrak if we do not specify tle_file

satellites = [
    Satellite("LOFT YAM-3", Orbital("YAM-3", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-5", Orbital("YAM-5", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-6", Orbital("YAM-6", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-7", Orbital("YAM-7", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-8", Orbital("YAM-8", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-10", Orbital("YAM-10", tle_file="TLEs.txt")),
    Satellite("Ubotica CogniSat-6 HAMMER", Orbital("HAMMER", tle_file="TLEs.txt")),
    # "LOFT YAM-3": Orbital("YAM-3"),
]

In [ ]:

ground_stations = [
    Location(
        -79.55,
        8.9833,
        0.028,
        "KSAT Panama",
    ),
    Location(
        -51.73363,
        64.182789,
        0,
        "KSAT Nuuk",
    ),
    Location(
        2.53219,
        -72.01243,
        0,
        "KSAT Troll",
    ),
    Location(
        142.3689,
        43.8,
        0,
        "KSAT Hokkaido",
    ),
    Location(
        103.9915,
        1.3661,
        0,
        "KSAT Singapore",
    ),
    Location(
        -70.85021,
        -52.93279,
        0,
        "KSAT Punta Arenas",
    ),
    Location(
        127.7766,
        26.4055,
        0,
        "KSAT Okinawa",
    ),
    Location(
        57.5565,
        -20.1142,
        0,
        "KSAT Mauritius",
    ),
    Location(
        22.62216,
        37.84604,
        0,
        "KSAT Nemea",
    ),
    Location(
        31.12509,
        70.36779,
        0,
        "KSAT Vardo",
    ),
    Location(
        15.39964,
        78.22875,
        0,
        "KSAT Svalbard",
    ),
]

# ground_station_opportunities = [
#     observation_request(
#         lon_deg=gs.lon_deg,
#         lat_deg=gs.lat_deg,
#         alt_km=gs.alt_km,
#         min_time=dt.datetime.now(dt.timezone.utc),
#         max_time=dt.datetime.now(dt.timezone.utc)+ dt.timedelta(seconds=3600*24*2)
#     )
#     for gs in ground_stations
# ]

In [ ]:
stride_s = 60
plot_range_s = 10800

# min_time = dt.datetime.now(dt.timezone.utc).replace(tzinfo=None)

min_time = dt.datetime(2026, 1, 14, 16, 55, 36, 841169)
max_time = min_time + dt.timedelta(hours=24)

In [ ]:


cities_of_the_world = pd.read_csv("simplemaps_worldcities_basicv1.901/worldcities.csv")
sampled_world_cities = cities_of_the_world.sample(n=100,weights='population',axis=0, random_state=0)
one_hundred_sampled_cities = [
    ObservationRequest(city[1].lng, city[1].lat, min_time=min_time, max_time=max_time, alt_km=0.307, request_name=city[1].city,)
    for city in sampled_world_cities.iterrows()
]

# Simulation

Let's talk about simulation.

We want an event-based sim. There is a global ordered timeline and the sim jumps from event to event.

Continuous transitions (power, thermal, etc) are discretized in this setting.

We need a few entities here.

Agent: a satellite, a constellation manager, a requestor. 

Event: something on the ground turning on or off.

Communication: something that affects the state of two entities.

Observation: a query of an agent to (event, empty set). If we want to be fancy, a query of an agent to a region, where events are or are 
not associated with regions.

Prototype:
- A Satellite class with a list of Observations
- A ConstellationManager class with a list of Satellites which can query Observations and add new ones.
  - A ConstellationManager class that updates the Satellites' Observations when a Communication event occurs.
- An Observation of a Region.
    - Observe. Returns an image of the region.
    - Search. Returns True if there is an Event in the Region. The event is stored.
    - Monitor. Returns True if the event is still active.
    - Something for moving events?
- Phenomenon: something with a spatial position, start time, end time, potentially internal states.

The simulator holds:
- Agents (Satellites, ConstellationManagers)
- Phenomena.
- A list of Events (Observation, Communication, Phenomenon transitions), encoded as functions.
The output of the functions affects the agents and phenomena.

We need to define:

- An Observation Opportunity (observation_opportunity)
- An Observer, which has an Orbit, a list of Observation Opportunities to execute, a list of ObservedEvents it has observed (with their state), and a list of DataProducts.
- A ConstellationManager, which has a list of ObserverStates (Orbit, ObservationOpportunities) and a list of CommunicationOpportunities with Observers. When there is a CommunicationOpportunity the ConstellationManager can push an updated list of ObservationOpportunities.
- An Event, with a lonlatalt, a State (enum), and times for StateTransitions.
- 

Something simple. 
- [X] Create a world.
- [X] Add satellites to it.
- [X] Add observation opportunities to the satellites manually
- [X] Click until out of events
- [X] Add a better way to add observation opportunities!
- [X] Add a ConstellationScheduler 

In [ ]:
phenomena = [
    Phenomenon(
        lon_deg = -118.,
        lat_deg = 34.,
        alt_km=0.307,
        start_time=min_time,
        end_time=max_time,

    ),
    Phenomenon(8., 45.,  alt_km=0.216, start_time=min_time, end_time=max_time),
    Phenomenon(-80., 34., alt_km=0.041, start_time=min_time, end_time=max_time),
]

In [ ]:
phenomena_cities = [
    Phenomenon(
        lon_deg = city.lon_deg,
        lat_deg = city.lat_deg,
        alt_km=city.alt_km,
        start_time=city.min_time,
        end_time=city.max_time,
        name=city.name
    ) for city in one_hundred_sampled_cities
]

In [ ]:
one_hundred_sampled_cities[0].min_time

In [ ]:
satellite_agents_simple = [
    Satellite("LOFT YAM-3", Orbital("YAM-3", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-5", Orbital("YAM-5", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-6", Orbital("YAM-6", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-7", Orbital("YAM-7", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-8", Orbital("YAM-8", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-10", Orbital("YAM-10", tle_file="TLEs.txt")),
    Satellite("Ubotica CogniSat-6 HAMMER", Orbital("HAMMER", tle_file="TLEs.txt")),
    # "LOFT YAM-3": Orbital("YAM-3"),
]

In [ ]:
world = World(satellites = satellite_agents_simple, phenomena=phenomena)

Manually add observation opportunities

In [ ]:
obs_requests = [
    ObservationRequest(-118., 34., min_time=min_time, max_time=max_time, alt_km=0.307),
    ObservationRequest(8., 45., min_time=min_time, max_time=max_time, alt_km=0.216),
    ObservationRequest(-80., 34., min_time=min_time, max_time=max_time, alt_km=0.041),
]

In [ ]:
observation_opportunities = find_observation_opportunities(obs_requests, satellites)

In [ ]:
best_request = {r: None for r in obs_requests}

for request, passes in observation_opportunities.items():
    _best_quality = - np.inf
    _best_satellite = None
    _best_pass = None
    for satellite, satpasses in passes.items():
        for satpass in satpasses:
            _quality = observation_quality(satpass.highest)
            if _quality >= _best_quality:
                _best_quality = _quality
                _best_satellite = satellite
                _best_pass = satpass
            # print("{}: quality {}".format(satpass.highest, observation_quality(satpass.highest)))
    best_request[request] = (_best_satellite, _best_pass)

In [ ]:
for req, opp in best_request.items():
    for _sat in world.satellites:
        if _sat.name == opp[0].name:
            _opportunity = opp[1].highest
            print("{} {} {}".format(_sat.name, req, _opportunity))

            schedule_observation(world, _sat, _opportunity)

In [ ]:
retcode = 1
while (retcode !=0):
    print("\nTick!")
    print(world.events)
    # print({sat.name: sat.scheduled_observations for sat in world.agents if len(sat.scheduled_observations)})
    retcode = world.tick()

In [ ]:
[sat.known_phenomena for sat in world.satellites]

In [ ]:
satellite_agents_world = [
    Satellite("LOFT YAM-3", Orbital("YAM-3", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-5", Orbital("YAM-5", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-6", Orbital("YAM-6", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-7", Orbital("YAM-7", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-8", Orbital("YAM-8", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-10", Orbital("YAM-10", tle_file="TLEs.txt")),
    Satellite("Ubotica CogniSat-6 HAMMER", Orbital("HAMMER", tle_file="TLEs.txt")),
    # "LOFT YAM-3": Orbital("YAM-3"),
]

In [ ]:
world_with_cities = World(satellites = satellite_agents_world, phenomena=phenomena_cities)



In [ ]:
best_request_cities = {r: None for r in one_hundred_sampled_cities}

observation_opportunities_cities = find_observation_opportunities(one_hundred_sampled_cities, satellite_agents_world)

for request, passes in observation_opportunities_cities.items():
    _best_quality = - np.inf
    _best_satellite = None
    _best_pass = None
    for satellite, satpasses in passes.items():
        for satpass in satpasses:
            _quality = observation_quality(satpass.highest)
            if _quality >= _best_quality:
                _best_quality = _quality
                _best_satellite = satellite
                _best_pass = satpass
            # print("{}: quality {}".format(satpass.highest, observation_quality(satpass.highest)))
    best_request_cities[request] = (_best_satellite, _best_pass)

In [ ]:
for req, opp in best_request_cities.items():
    for _sat in world_with_cities.satellites:
        # Give the event to the right agent
        if _sat.name == opp[0].name:
            # The opportunity is the middle of the pass
            _opportunity = opp[1].highest
            # print("{} {} {}".format(_sat.name, req, _opportunity))

            # This is quite redundant. What you want is to maintain events for individual agents and then a global copy, right?
            _sat.scheduled_observations.append(_opportunity)
            _event = Event(
                name="Observation for {}: {}".format(opp[0].name, opp[1]),
                time = _opportunity.time,
                # The magic is here: we add an event that does an observation and stores the result in the agent's known_phenomena bin
                # Note the kludge of default inputs to make sure the closure works and we capture the variables at the time of creation
                # action_callable = lambda _opp=_opportunity, _satname=_sat.name: print("{} with {}".format(_opp, _satname)) #_sat.known_phenomena.append(world.do_observation(_opportunity, _sat))
                action_callable = lambda _opp=_opportunity, _sate=_sat: world_with_cities.do_observation(_opp, _sate)
            )
            world_with_cities.add_event(_event)

In [ ]:
retcode = 1
while (retcode !=0):
    # print("Tick!")
    # print(world_cities.events)
    # print({sat.name: sat.scheduled_observations for sat in world.agents if len(sat.scheduled_observations)})
    retcode = world_with_cities.tick()

In [ ]:
satellite_agents_sched = [
    Satellite("LOFT YAM-3", Orbital("YAM-3", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-5", Orbital("YAM-5", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-6", Orbital("YAM-6", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-7", Orbital("YAM-7", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-8", Orbital("YAM-8", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-10", Orbital("YAM-10", tle_file="TLEs.txt")),
    Satellite("Ubotica CogniSat-6 HAMMER", Orbital("HAMMER", tle_file="TLEs.txt")),
    # "LOFT YAM-3": Orbital("YAM-3"),
]

world_with_scheduler = World(satellites = satellite_agents_sched, phenomena=phenomena)

scheduler = ConstellationGroundScheduler(satellites=satellite_agents_sched, ground_stations=ground_stations, world=world_with_scheduler)

world_with_scheduler.add_constellation(scheduler)

In [ ]:
for request in obs_requests:
    scheduler.schedule_request(
        request=request,
        current_time=dt.datetime.now(dt.timezone.utc).replace(tzinfo=None),
        callback_request_scheduled=lambda obs: print("Request {} scheduled for observation {}!".format(request, obs)),
        callback_request_ready=lambda dp: print("Request {} ready with DP {}!".format(request, dp)),
        )

In [ ]:
scheduler.schedule_downlinks()

In [ ]:
retcode = 1
while (retcode !=0):
    # print("Tick!")
    retcode = world_with_scheduler.tick(print_forbidden_prefixes=["Downlink", "End of downlink"])
    # print(world_with_scheduler.events)

In [ ]:
# scheduler.requests

In [ ]:
scheduler._requests

Where do we go from here?

The obvious: add conflicts between observations. For now, we can stay in discrete land where an opportunity is an instantaneous thing. Or we can create multiple copies all along the path.
Conflict: two opportunities are on the same satellite and closer than some amount of time. The time should account for (i) some fixed setup (think "wait for the mirror to stop flapping") plus some time related to reorientation.
This misses the fact that one could point the spacecraft slightly away from the target and still get it. 
Can we get that in pre-processing?

Solve the one-off scheduling problem with constraints (greedy, find the best observation that is feasible).

Solve the one-off scheduling problem with constraints including comms (greedy, find the best observation that is feasible after we can talk to a given satellite).

Solve the batch scheduling problem with constraints (ILP? For old times' sake).

Multiple instruments. Spacecraft should have an instrument attached. Requests and S/C have an instrument.

Follow-on requests. A detection causes a follow-up request. Simulate.

Write up the three cases of interest:
- Submit a request and you immediately hear back. Unsubmitting requests is free.
    - Use your favorite black-box scheduling algorithm. Submitting a request==evaluating a constraint.
- Submit a request and you immediately hear back. Unsubmitting requests is expensive.
    - Use your favorite non-backtracking scheduling algorithm. Once you choose, no regrets.
- Submit a request and you don't hear back. This is a DMU problem.
    - State:
    - Actions: schedule an observation on a satellite.
    - Transitions: from "unscheduled" to "scheduled" to "executed" or "rejected" for every observation. 
    - Observations: when a file is downloaded, we find out.
    - Rewards: k if we get an observation, 0 otherwise.

- [ ] Constellation: 
  - Input:
      - [X] A new request
      - [X] A schedule of observations assigned to agents
      - [X] (can compute) Alternate windows for the assigned observations
  - Output:
      - A new schedule of observations assigned to agents
      - [X] A bool indicating whether the request was assigned
  - Formulate the optimization problem:
      - Identify which observations can be unscheduled (are not yet committed)
      - Formulate the profit-maximizing problem of assigning everything
      - Solve the problem
      - Check if the new observation is in.
- [ ] Broker
  - Input:
      - A new request
  - Output:
      - Time when the request is scheduled
  - List all windows across all constellations
  - Pick the best one
  - Send the request
- API:
    - [ ] Constellation
        - Input:
            - [X] Submit a request
            - [X] Retrieve a request status
        - Output:
            - [X] Report data acquired
            - [X] Report event
            - [X] Report unscheduled task
    - [ ] Broker
        - Input:
            - Submit a workflow request
            - Status update on observation request
        - Output
            -  Observation requests
            -  Status updates on workflow requests

In [ ]:
# Let's have multiple constellations!

# Behind the scenes, this pulls from Celestrak if we do not specify tle_file

satellites_LOFT = [
    # Satellite("LOFT YAM-3", Orbital("YAM-3", tle_file="TLEs.txt")),
    # Satellite("LOFT YAM-5", Orbital("YAM-5", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-6", Orbital("YAM-6", tle_file="TLEs.txt")),
    # Satellite("LOFT YAM-7", Orbital("YAM-7", tle_file="TLEs.txt")),
    # Satellite("LOFT YAM-8", Orbital("YAM-8", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-10", Orbital("YAM-10", tle_file="TLEs.txt")),
    # Satellite("Ubotica CogniSat-6 HAMMER", Orbital("HAMMER", tle_file="TLEs.txt")),
    # "LOFT YAM-3": Orbital("YAM-3"),
]

satellites_ubotica = [
    Satellite("Ubotica CogniSat-6 HAMMER", Orbital("HAMMER", tle_file="TLEs.txt")),
    ]

satellites_aerospace = [
        Satellite("AEROCUBE 18A", Orbital("AEROCUBE 18A", tle_file="TLEs.txt")),
        Satellite("AEROCUBE 18B", Orbital("AEROCUBE 18B", tle_file="TLEs.txt")),   

]

satellites_capella = [
        Satellite("AEROCUBE 18A", Orbital("AEROCUBE 18A", tle_file="TLEs.txt")),
        Satellite("AEROCUBE 18B", Orbital("AEROCUBE 18B", tle_file="TLEs.txt")),   

]

satellites_multischedulers = satellites_LOFT+satellites_ubotica+satellites_aerospace

In [ ]:
world_with_schedulers = World(satellites = satellites_multischedulers, phenomena=phenomena_cities)

scheduler_LOFT = ConstellationGroundScheduler(satellites=satellites_LOFT, ground_stations=ground_stations, world=world_with_schedulers, name="LOFT")
scheduler_ubotica = ConstellationGroundScheduler(satellites=satellites_ubotica, ground_stations=ground_stations, world=world_with_schedulers, name="UBOTICA")
scheduler_aerospace = ConstellationGroundScheduler(satellites=satellites_aerospace, ground_stations=ground_stations, world=world_with_schedulers, name="AC")

world_with_schedulers.add_constellation(scheduler_LOFT)
world_with_schedulers.add_constellation(scheduler_ubotica)
world_with_schedulers.add_constellation(scheduler_aerospace)

In [ ]:
sampled_world_cities_LOFT = cities_of_the_world.sample(n=50,weights='population',axis=0, random_state=0)
sampled_world_cities_ubotica = cities_of_the_world.sample(n=50,weights='population',axis=0, random_state=1)
sampled_world_cities_aerospace = cities_of_the_world.sample(n=50,weights='population',axis=0, random_state=2)

obs_request_LOFT = [
    ObservationRequest(city[1].lng, city[1].lat, min_time=min_time, max_time=max_time, alt_km=0.307, request_name=city[1].city,)
    for city in sampled_world_cities_LOFT.iterrows()
]

obs_request_ubotica = [
    ObservationRequest(city[1].lng, city[1].lat, min_time=min_time, max_time=max_time, alt_km=0.307, request_name=city[1].city,)
    for city in sampled_world_cities_ubotica.iterrows()
]

obs_request_aerospace = [
    ObservationRequest(city[1].lng, city[1].lat, min_time=min_time, max_time=max_time, alt_km=0.307, request_name=city[1].city,)
    for city in sampled_world_cities_aerospace.iterrows()
]

In [ ]:
# sim_start_time = dt.datetime.now(dt.timezone.utc).replace(tzinfo=None)
sim_start_time = dt.datetime(2026, 1, 14, 20, 22, 55, 552917)

for request in obs_request_LOFT:
    scheduler_LOFT.schedule_request(
        request=request,
        current_time=sim_start_time,
        callback_request_scheduled=lambda obs: print("[LOFT] Request {} scheduled for observation {}!".format(request, obs)),
        callback_request_ready=lambda dp: print("[LOFT] Request {} ready with DP {}!".format(request, dp)),
        )
    
for request in obs_request_ubotica:
    scheduler_ubotica.schedule_request(
        request=request,
        current_time=sim_start_time,
        callback_request_scheduled=lambda obs: print("[UBOTICA] Request {} scheduled for observation {}!".format(request, obs)),
        callback_request_ready=lambda dp: print("[UBOTICA] Request {} ready with DP {}!".format(request, dp)),
        )
    
for request in obs_request_aerospace:
    scheduler_aerospace.schedule_request(
        request=request,
        current_time=sim_start_time,
        callback_request_scheduled=lambda obs: print("[AC] Request {} scheduled for observation {}!".format(request, obs)),
        callback_request_ready=lambda dp: print("[AC] Request {} ready with DP {}!".format(request, dp)),
        )

# for request in obs_request_LOFT:
#     scheduler_LOFT.schedule_request(
#         request=request,
#         current_time=sim_start_time,
#         callback_request_scheduled=lambda obs: print("[LOFT] Request {} scheduled for observation {}!".format(request, obs)),
#         callback_request_ready=lambda dp: print("[LOFT] Request {} ready with DP {}!".format(request, dp)),
#         )
    
# for request in obs_request_ubotica:
#     scheduler_ubotica.schedule_request(
#         request=request,
#         current_time=sim_start_time,
#         callback_request_scheduled=lambda obs: print("[UBOTICA] Request {} scheduled for observation {}!".format(request, obs)),
#         callback_request_ready=lambda dp: print("[UBOTICA] Request {} ready with DP {}!".format(request, dp)),
#         )
    
# for request in obs_request_aerospace:
#     scheduler_aerospace.schedule_request(
#         request=request,
#         current_time=sim_start_time,
#         callback_request_scheduled=lambda obs: print("[AC] Request {} scheduled for observation {}!".format(request, obs)),
#         callback_request_ready=lambda dp: print("[AC] Request {} ready with DP {}!".format(request, dp)),
#         )

In [ ]:
# Downlinks are now built-in to observations!

# scheduler_LOFT.schedule_downlinks(current_time=sim_start_time, max_time=sim_start_time+dt.timedelta(hours=36))
# scheduler_ubotica.schedule_downlinks(current_time=sim_start_time, max_time=sim_start_time+dt.timedelta(hours=36))
# scheduler_aerospace.schedule_downlinks(current_time=sim_start_time, max_time=sim_start_time+dt.timedelta(hours=36))

In [ ]:
world_with_schedulers.events

In [ ]:
retcode = 1
while (retcode !=0):
    retcode = world_with_schedulers.tick()
    # retcode = world_with_schedulers.tick(print_forbidden_prefixes=["Downlink", "End of downlink", "Unlock uplink", "Unlock satellite after obs"])

In [ ]:
def request_statistics(requests_pd):
    total_requests_no = len(requests_pd)
    all_statuses = set(requests_pd.status.values)    
    for s in all_statuses:
        # matching_statuses = sum([1 if (r['status']==s) else 0 for r in requests.values()])
        matching_statuses = len(requests_pd[requests_pd.status==s])
        print("{}/{} ({}%) of requests are in status {}".format(matching_statuses,total_requests_no, matching_statuses/total_requests_no*100, s))
    # X/Y requests have >1 successful observation
    unique_requests = set(requests_pd.request)
    unique_requests_no = len(unique_requests)
    fulfilled_unique_requests_no = 0
    for ur in unique_requests:
        matching_observation_statuses = requests_pd[(requests_pd['request']==ur) & (requests_pd['status']=="OK! Data received")]
        if len(matching_observation_statuses):
            fulfilled_unique_requests_no += 1
    print(" {}/{} ({}%) unique requests have at least one successful observation".format(fulfilled_unique_requests_no, unique_requests_no, fulfilled_unique_requests_no/unique_requests_no*100))

In [ ]:
print("LOFT")
request_statistics(scheduler_LOFT._requests)

print("Ubotica")
request_statistics(scheduler_ubotica._requests)

print("AC")
request_statistics(scheduler_aerospace._requests)




In [ ]:
# class Broker():
#     def __init__(self, constellations: list[ConstellationGroundScheduler], world: World, name="Broker"):
#         self.name = name
#         self.constellations = constellations
#         # self.known_satellites = known_satellites
#         self.world = world
#         self._requests = pd.DataFrame(columns=['request', 'requested_pass', 'requested_constellation', 'requested_satellite', 'constellation', 'satellite', 'assigned_pass', 'assigned_downlink', 'status', 'data_product', 'scheduled_callback', 'unscheduled_callback', 'ready_callback'])

#         # self.requests = {}

#     def _screen_pass_for_feasibility(self, satellite: Satellite, _obs_pass: ObservationPass):
#         # Check if a given pass conflicts with existing requests.
#         # TODO this is horrifyingly expensive because we do not exploit the fact that
#         #  requests are sorted. We should improve this, ideally without rebuilding a full on timeline library.
#         if len(self._requests):
#             conflicting_requests = self._requests.loc[
#                 self._requests.apply(
#                 lambda x: 
#                     (x['status'] != "OK! Data received") and # We have submitted this, or it's scheduled, OR IT FAILED TO SCHEDULE (which suggests this is a bad time)
#                     (x['requested_pass'] is not None) and
#                     (x['requested_satellite'] is not None) and
#                     (x['requested_pass'].highest.time+x['requested_pass'].highest.duration > _obs_pass.rise.time) and # The end of the other observation is after we start
#                     (x['requested_pass'].highest.time < _obs_pass.fall.time) and # The start of the other observation is before we end
#                     (x['requested_satellite'] == satellite) # This request is on the same satellite. Note that we check these are the same OBJECT, not just the same name.
#                 , axis=1)]
#             if len(conflicting_requests):
#                 return False
#         return True

#     # Broadly, look at the ephemerides, find the best option, find the corresponding constellation, give them a window around that.
#     def schedule_request(
#             self,
#             request: ObservationRequest,
#             current_time: dt.datetime=dt.datetime.now(dt.timezone.utc).replace(tzinfo=None),
#             number_of_submissions: int=1,
#             ):
#         # Pick the best satellite to fulfill this. This is where we'll need to be smarter. Or not! Just pick something starting the day after.
#         print("[{}] scheduling request {}".format(self.name, request))
#         # self.requests[request] = {


#         _known_satellites = []
#         _known_satellites_by_constellation = {}
#         for constellation in self.constellations:
#             _known_satellites += constellation.satellites
#             for _sat in constellation.satellites:
#                 _known_satellites_by_constellation[_sat] = constellation

#         _opportunities = find_observation_opportunities(
#             [request,],
#             satellites=_known_satellites,
#             passes_error_s=60,
#             passes_horizon_deg=MIN_HORIZON_ANGLE_FOR_OBS_DEG
#         )
#         # self.requests[request]['opportunities'] = _opportunities
#         # self._requests.loc[self._requests['request']==request, 'opportunities'] = _opportunities

#         passes = _opportunities[request]

#         if len(passes):
            
#             sorted_passes = [(satellite, satpass) for satellite, satpasses in passes.items() for satpass in satpasses]
#             # Sort by quality
#             sorted_passes.sort(key=lambda x: observation_quality(x[1].highest), reverse=True)
            
#             _best_satellite = sorted_passes[0][0]
#             _best_pass = sorted_passes[0][1]
#             _best_quality = observation_quality(_best_pass.highest)

#             print("Best request: {} with {}".format(_best_pass, _best_satellite))
#         else:
#             print("No observation opportunities here")
#             # self.requests[request]['status'] = "No observation opportunities";
#             _request_dict = {
#                 'request': request,
#                 'requested_pass' : None,
#                 'requested_constellation' : None,
#                 'requested_satellite' : None,
#                 'constellation': None,
#                 'satellite': None,
#                 'assigned_pass': None,
#                 'assigned_downlink': None,
#                 'status': "No observation opportunities",
#                 'data_product': None,
#                 'scheduled_callback': lambda x: None,
#                 'unscheduled_callback': lambda x: None,
#                 'ready_callback': lambda x: None,
#             }
#             _pdrequest = pd.DataFrame([_request_dict])
#             self._requests = pd.concat([self._requests, _pdrequest], ignore_index=True)

#             # self._requests.loc[self._requests['request']==request, 'status'] = "No observation opportunities"
#             return -1

#         successful_submissions_for_this_request = 0

#         for opportunity_ix in range(len(sorted_passes)): # Odd legacy construction, we should probably iterate directly
#             if successful_submissions_for_this_request>=number_of_submissions:
#                 break

#             _best_pass = sorted_passes[opportunity_ix][1]
#             _best_satellite = sorted_passes[opportunity_ix][0]

#             if (_best_pass is not None) and (_best_satellite is not None):

#                 if not (self._screen_pass_for_feasibility(satellite, _best_pass)):
#                     # This pass is not feasible, forget about it
#                     print(" Broker skipping a good pass for feasibility")
#                     continue
            
#                 # We are going to register a submission for this
#                 successful_submissions_for_this_request += 1

#                 _best_constellation = _known_satellites_by_constellation[_best_satellite]

#                 def callback_request_scheduled(assigned_pass, _request=request, __best_pass=_best_pass, __best_constellation=_best_constellation, __best_satellite=_best_satellite):
#                     print(" [{}] confirmed scheduling of request {} from pass {}, constellation {}".format(self.name, _request, __best_pass, __best_constellation.name))
#                     self._requests.loc[((self._requests['request']==_request) & (self._requests['requested_pass']==__best_pass)), 'assigned_pass'] = assigned_pass
#                     self._requests.loc[((self._requests['request']==_request) & (self._requests['requested_pass']==__best_pass)), 'status'] = "Scheduled"
#                     self._requests.loc[((self._requests['request']==_request) & (self._requests['requested_pass']==__best_pass)), 'constellation'] = __best_constellation
#                     self._requests.loc[((self._requests['request']==_request) & (self._requests['requested_pass']==__best_pass)), 'satellite'] = __best_satellite
#                     # TODO Mark this satellite/pass as a busy time, keep track for internal rescheduling
#                     return
                
#                 def callback_request_unscheduled(reason, _request=request, __best_pass=_best_pass, __best_constellation=_best_constellation):
#                     print(" [{}] received UNscheduling of request {}, pass {}, from {}".format(self.name, request, _best_pass, _best_constellation.name))
#                     self._requests.loc[((self._requests['request']==_request) & (self._requests['requested_pass']==__best_pass)), 'assigned_pass'] = None
#                     self._requests.loc[((self._requests['request']==_request) & (self._requests['requested_pass']==__best_pass)), 'status'] = reason
#                     self._requests.loc[((self._requests['request']==_request) & (self._requests['requested_pass']==__best_pass)), 'constellation'] = None
#                     self._requests.loc[((self._requests['request']==_request) & (self._requests['requested_pass']==__best_pass)), 'satellite'] = None
#                     # TODO Mark this satellite/pass as a bad time, keep track for internal rescheduling
#                     # Also reschedule

#                     return
                
#                 def callback_request_ready(data_product,  _request=request, __best_pass=_best_pass, __best_constellation=_best_constellation):
#                     print(" [{}: ] data ready for request {}, pass {}, from {}".format(self.name, _request, __best_pass, __best_constellation.name))
#                     self._requests.loc[((self._requests['request']==_request) & (self._requests['requested_pass']==__best_pass)), 'status'] = "OK! Data received"
#                     self._requests.loc[((self._requests['request']==_request) & (self._requests['requested_pass']==__best_pass)), 'data_product'] = data_product
#                     # TODO Attempt to cancel other requests for this observation
#                     return

#                 _constellation_request = ObservationRequest(
#                     lon_deg=request.lon_deg,
#                     lat_deg=request.lat_deg,
#                     min_time=_best_pass.rise.time-dt.timedelta(minutes=1), # This is the magic, we constrain the request to the constellation AND TIME that we like.
#                     max_time=_best_pass.fall.time+dt.timedelta(minutes=1),
#                     alt_km=request.alt_km,
#                     instrument=request.instrument,
#                     request_name=request.name,
#                 )

#                 _request_dict = {
#                     'request': request,
#                     'requested_pass' : _best_pass,
#                     'requested_constellation' : _best_constellation,
#                     'requested_satellite' :_best_satellite,
#                     'constellation': None,
#                     'satellite': None,
#                     'assigned_pass': None,
#                     'assigned_downlink': None,
#                     'status': "Submitted",
#                     'data_product': None,
#                     'scheduled_callback': lambda x: None,
#                     'unscheduled_callback': lambda x: None,
#                     'ready_callback': lambda x: None,
#                 }
#                 _pdrequest = pd.DataFrame([_request_dict])
#                 self._requests = pd.concat([self._requests, _pdrequest], ignore_index=True)

#                 # Submit the request to the relevant constellation
#                 _best_constellation.schedule_request(
#                     request=_constellation_request,
#                     current_time=current_time,
#                     callback_request_scheduled=callback_request_scheduled,
#                     callback_request_unscheduled=callback_request_unscheduled,
#                     callback_request_ready=callback_request_ready,
#                 )
#         if successful_submissions_for_this_request == 0:
#             print("No unconflicted opportunities here")
#             # self.requests[request]['status'] = "No observation opportunities";
#             _request_dict = {
#                 'request': request,
#                 'requested_pass' : None,
#                 'requested_constellation' : None,
#                 'requested_satellite' : None,
#                 'constellation': None,
#                 'satellite': None,
#                 'assigned_pass': None,
#                 'assigned_downlink': None,
#                 'status': "No unconflicted observation opportunities",
#                 'data_product': None,
#                 'scheduled_callback': lambda x: None,
#                 'unscheduled_callback': lambda x: None,
#                 'ready_callback': lambda x: None,
#             }
#             _pdrequest = pd.DataFrame([_request_dict])
#             self._requests = pd.concat([self._requests, _pdrequest], ignore_index=True)
                


#     # Do the silly thing: decompose the workflow deterministically, then assign ALL those requests...
#     def schedule_workflow(
#             self,
#             workflow,
#             current_time: dt.datetime=dt.datetime.now(dt.timezone.utc).replace(tzinfo=None),
#     ):
#         pass





In [ ]:
sim_start_time = dt.datetime.now(dt.timezone.utc).replace(tzinfo=None)

satellites_LOFT = [
    Satellite("LOFT YAM-6", Orbital("YAM-6", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-10", Orbital("YAM-10", tle_file="TLEs.txt")),
]

satellites_ubotica = [
    Satellite("Ubotica CogniSat-6 HAMMER", Orbital("HAMMER", tle_file="TLEs.txt")),
    ]

satellites_aerospace = [
        Satellite("AEROCUBE 18A", Orbital("AEROCUBE 18A", tle_file="TLEs.txt")),
        Satellite("AEROCUBE 18B", Orbital("AEROCUBE 18B", tle_file="TLEs.txt")),   
]

satellites_multischedulers = satellites_LOFT+satellites_ubotica+satellites_aerospace

In [ ]:
world_with_brokers = World(satellites = satellites_multischedulers, phenomena=phenomena_cities)

N_BACKGROUND_SAMPLES_PER_CONSTELLATION = 50

scheduler_LOFT = ConstellationGroundScheduler(satellites=satellites_LOFT, ground_stations=ground_stations, world=world_with_brokers, name="LOFT")
scheduler_ubotica = ConstellationGroundScheduler(satellites=satellites_ubotica, ground_stations=ground_stations, world=world_with_brokers, name="UBOTICA")
scheduler_aerospace = ConstellationGroundScheduler(satellites=satellites_aerospace, ground_stations=ground_stations, world=world_with_brokers, name="AC")

world_with_brokers.add_constellation(scheduler_LOFT)
world_with_brokers.add_constellation(scheduler_ubotica)
world_with_brokers.add_constellation(scheduler_aerospace)

sampled_world_cities_LOFT = cities_of_the_world.sample(n=N_BACKGROUND_SAMPLES_PER_CONSTELLATION,weights='population',axis=0, random_state=0)
sampled_world_cities_ubotica = cities_of_the_world.sample(n=N_BACKGROUND_SAMPLES_PER_CONSTELLATION,weights='population',axis=0, random_state=1)
sampled_world_cities_aerospace = cities_of_the_world.sample(n=N_BACKGROUND_SAMPLES_PER_CONSTELLATION,weights='population',axis=0, random_state=2)
obs_request_LOFT = [
    ObservationRequest(city[1].lng, city[1].lat, min_time=min_time, max_time=max_time, alt_km=0.307, request_name=city[1].city,)
    for city in sampled_world_cities_LOFT.iterrows()
]
obs_request_ubotica = [
    ObservationRequest(city[1].lng, city[1].lat, min_time=min_time, max_time=max_time, alt_km=0.307, request_name=city[1].city,)
    for city in sampled_world_cities_ubotica.iterrows()
]
obs_request_aerospace = [
    ObservationRequest(city[1].lng, city[1].lat, min_time=min_time, max_time=max_time, alt_km=0.307, request_name=city[1].city,)
    for city in sampled_world_cities_aerospace.iterrows()
]


for request in obs_request_LOFT:
    scheduler_LOFT.schedule_request(
        request=request,
        current_time=sim_start_time,
        callback_request_scheduled=lambda obs: print("[LOFT] Request {} scheduled for observation {}!".format(request, obs)),
        callback_request_ready=lambda dp: print("[LOFT] Request {} ready with DP {}!".format(request, dp)),
        )
    
for request in obs_request_ubotica:
    scheduler_ubotica.schedule_request(
        request=request,
        current_time=sim_start_time,
        callback_request_scheduled=lambda obs: print("[UBOTICA] Request {} scheduled for observation {}!".format(request, obs)),
        callback_request_ready=lambda dp: print("[UBOTICA] Request {} ready with DP {}!".format(request, dp)),
        )
    
for request in obs_request_aerospace:
    scheduler_aerospace.schedule_request(
        request=request,
        current_time=sim_start_time,
        callback_request_scheduled=lambda obs: print("[AC] Request {} scheduled for observation {}!".format(request, obs)),
        callback_request_ready=lambda dp: print("[AC] Request {} ready with DP {}!".format(request, dp)),
        )
    
# scheduler_LOFT.schedule_downlinks(current_time=sim_start_time, max_time=sim_start_time+dt.timedelta(hours=36))
# scheduler_ubotica.schedule_downlinks(current_time=sim_start_time, max_time=sim_start_time+dt.timedelta(hours=36))
# scheduler_aerospace.schedule_downlinks(current_time=sim_start_time, max_time=sim_start_time+dt.timedelta(hours=36))

In [ ]:
broker = Broker(constellations=[scheduler_LOFT, scheduler_ubotica, scheduler_aerospace], world=world_with_brokers)

world_with_brokers.add_broker(broker)

In [ ]:
sampled_world_cities_broker = cities_of_the_world.sample(n=50,weights='population',axis=0, random_state=3)

number_of_submissions = 5

obs_requests_broker = [
    ObservationRequest(city[1].lng, city[1].lat, min_time=min_time, max_time=max_time, alt_km=0.307, request_name=city[1].city,)
    for city in sampled_world_cities_broker.iterrows()
]

for _req in obs_requests_broker:
    broker.schedule_request(_req, current_time=sim_start_time, number_of_submissions=number_of_submissions)

In [ ]:
retcode = 1
while (retcode !=0):
    # print("Tick!")
    retcode = world_with_brokers.tick(print_forbidden_prefixes=["Downlink", "End of downlink", "Unlock uplink", "Unlock satellite after obs"])
    # print(world_with_scheduler.events)

In [ ]:
request_statistics(broker._requests)

In [ ]:
def retell_history(world: World):
    for _chronicle in world.history:
        print("Time: {}. Event: {}".format(_chronicle['time'], _chronicle['event']))
        if type(_chronicle['event'])==ObservationEvent:
            print("Observation: sat {} and opportunity {}".format(_chronicle['event'].satellite, _chronicle['event'].opportunity))
        if type(_chronicle['event'])==CommunicationEvent:
            print("Communication: station {} to sat {} during pass {}".format(_chronicle['event'].station, _chronicle['event'].satellite, _chronicle['event'].comm_pass))

In [ ]:
retell_history(world_with_brokers)

In [ ]:
def plot_event(_chronicle: dict, world: World, ax=None):
    if ax is None:
        figglobal = plt.figure(figsize=(10,5))
        ax = figglobal.add_subplot(1,1,1, projection=ccrs.Robinson())
        ax.set_global()
        ax.coastlines()

    _time_to_plot_ground_track = dt.timedelta(seconds=15*60)
    _dt_to_plot_ground_track = dt.timedelta(seconds=60)
    time_steps_for_plotting = [_chronicle['time']- _dt_to_plot_ground_track*i for i in range(int(math.ceil(_time_to_plot_ground_track/_dt_to_plot_ground_track)))]

    ax.text(0,0,"{}".format(_chronicle['time']), transform=ax.transAxes)

    # for satellite in world.satellites:
    #     _orbit = satellite.orbit
    #     _llas = [_orbit.get_lonlatalt(t) for t in time_steps_for_plotting]
    #     # ax.plot([lla[0] for lla in _llas],[lla[1] for lla in _llas],transform=ccrs.Geodetic())
    #     ax.plot([lla[0] for lla in _llas],[lla[1] for lla in _llas],transform=ccrs.Geodetic())
    
    constellation_palette = cmap['viridis'].resampled(len(world.constellations))

    for constellation_ix, constellation in enumerate(world.constellations):
        constellation_color = constellation_palette(constellation_ix/len(world.constellations))
        for ground_station in constellation.ground_stations:
            ax.plot(float(ground_station.lon_deg), float(ground_station.lat_deg), '*', transform=ccrs.PlateCarree(), color=constellation_color)

        for satellite in constellation.satellites:
            _orbit = satellite.orbit
            _llas = [_orbit.get_lonlatalt(t) for t in time_steps_for_plotting]
            # ax.plot([lla[0] for lla in _llas],[lla[1] for lla in _llas],transform=ccrs.Geodetic())
            ax.plot([lla[0] for lla in _llas],[lla[1] for lla in _llas],transform=ccrs.Geodetic(), color=constellation_color)
    
    if type(_chronicle['event'])==ObservationEvent:
        ax.plot(
            [_chronicle['event'].opportunity.lon_deg, _chronicle['event'].satellite.orbit.get_lonlatalt(_chronicle['time'])[0]],
            [_chronicle['event'].opportunity.lat_deg, _chronicle['event'].satellite.orbit.get_lonlatalt(_chronicle['time'])[1]],
            ':k',
            transform=ccrs.Geodetic()
        )
        ax.plot(
            _chronicle['event'].opportunity.lon_deg,
            _chronicle['event'].opportunity.lat_deg,
            'Dr',
            markersize=10,
            transform=ccrs.Geodetic()
        )
        ax.plot(
            _chronicle['event'].satellite.orbit.get_lonlatalt(_chronicle['time'])[0],
            _chronicle['event'].satellite.orbit.get_lonlatalt(_chronicle['time'])[1],
            '.',
            markersize=10,
            transform=ccrs.Geodetic()
        )

    elif type(_chronicle['event'])==CommunicationEvent:
        ax.plot(
            [_chronicle['event'].station.lon_deg, _chronicle['event'].satellite.orbit.get_lonlatalt(_chronicle['time'])[0]],
            [_chronicle['event'].station.lat_deg, _chronicle['event'].satellite.orbit.get_lonlatalt(_chronicle['time'])[1]],
            '-.k',
            transform=ccrs.Geodetic()
        )
        ax.plot(
            _chronicle['event'].station.lon_deg,
            _chronicle['event'].station.lat_deg,
            '*',
            markersize=10,
            transform=ccrs.Geodetic()
        )
        ax.plot(
            _chronicle['event'].satellite.orbit.get_lonlatalt(_chronicle['time'])[0],
            _chronicle['event'].satellite.orbit.get_lonlatalt(_chronicle['time'])[1],
            '.',
            markersize=10,
            transform=ccrs.Geodetic()
        )
    return ax


def plot_history(world: World):
    artists = []
    for _chronicle_ix, _chronicle in enumerate(world.history):
        _ax = plot_event(_chronicle, world)
        plt.savefig("FRAME_{:05d}.png".format(_chronicle_ix))
        # artists.append(_ax)
        
    # plt.show()
        # if type(_chronicle['event'])==ObservationEvent:
        #     print("Observation: sat {} and opportunity {}".format(_chronicle['event'].satellite, _chronicle['event'].opportunity))
        # if type(_chronicle['event'])==CommunicationEvent:
        #     print("Communication: station {} to sat {} during pass {}".format(_chronicle['event'].station, _chronicle['event'].satellite, _chronicle['event'].comm_pass))

In [ ]:
# plot_history(world_with_brokers)
# # for i in range(10):
#     # plot_event(world_with_brokers.history[i], world_with_brokers)
